In [ ]:
!pip install transformers

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os
from datasets import load_dataset, Image

#Load the JSONL files
ds = load_dataset(
    "json",
    data_files={
        "train": "data_train.jsonl",
        "test": "data_val.jsonl",
    },
)

#Image directories
TRAIN_IMAGE_DIR = "train"
VAL_IMAGE_DIR = "test"

#Join the correct image directory to each filename
def add_train_path(example):
    example["image"] = os.path.join(TRAIN_IMAGE_DIR, example["image"])
    return example

def add_val_path(example):
    example["image"] = os.path.join(VAL_IMAGE_DIR, example["image"])
    return example

ds["train"] = ds["train"].map(add_train_path)
ds["test"] = ds["test"].map(add_val_path)

#Convert image paths into PIL Images
ds["train"] = ds["train"].cast_column("image", Image())
ds["test"] = ds["test"].cast_column("image", Image())

# Create the train/validation datasets
train_ds = ds["train"]
val_ds = ds["test"]

In [ ]:
from transformers import PaliGemmaProcessor
model_id = "google/paligemma-3b-pt-224"
processor = PaliGemmaProcessor.from_pretrained(model_id)

In [ ]:
import torch
device = "cuda"

image_token = processor.tokenizer.convert_tokens_to_ids("<image>")
def collate_fn(examples):
    images = [example["image"].convert("RGB") for example in examples]

    prompts = ["<image>What is this an image of?" for _ in examples]

    #Ground-truth class labels
    labels = [example["text"] for example in examples]

    tokens = processor(
        text=prompts,
        images=images,
        suffix=labels,
        return_tensors="pt",
        padding="longest",
    )

    return tokens

In [ ]:
from transformers import PaliGemmaForConditionalGeneration
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.bfloat16).to(device)

for param in model.model.vision_tower.parameters():
    param.requires_grad = False

for param in model.model.multi_modal_projector.parameters():
    param.requires_grad = True

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_type=torch.bfloat16
)

lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments
args=TrainingArguments(
            output_dir = "paligemma-finetuned-rhb",
            num_train_epochs=1,
            remove_unused_columns=False,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=0.0002,
            logging_steps=5,
            optim="adamw_torch_fused",
            save_strategy="epoch",
            bf16=True,
        )

In [ ]:
!pip install trl
from trl import SFTTrainer
trainer = SFTTrainer(
        model=model,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        args=args
        )
trainer.train()

In [ ]:
trainer.save_model()
processor.save_pretrained(args.output_dir)